# Player Performance Prediction: Model Evaluation & Metrics
This notebook extracts the regression and classification metrics for the Random Forest model and visualizes the results.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, fbeta_score, confusion_matrix
)
import joblib

# Visualization settings
plt.style.use('ggplot')
sns.set_palette('viridis')
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
# Load Data
df = pd.read_parquet('data/player_features.parquet')

features = ['last_game', 'last3_avg', 'last5_avg', 'last7_avg']
X = df[features]
y = df['rating']

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Load Model
model = joblib.load('models/rating_model.pkl')
preds = model.predict(X_test)

print('Data and Model successfully loaded.')

### 1. Regression Metrics & Visualization

In [ ]:
print('--- REGRESSION METRICS ---')
print(f'MAE:  {mean_absolute_error(y_test, preds):.4f}')
print(f'RMSE: {np.sqrt(mean_squared_error(y_test, preds)):.4f}')
print(f'R2:   {r2_score(y_test, preds):.4f}')

In [ ]:
plt.figure(figsize=(8, 8))
plt.scatter(y_test, preds, alpha=0.3, color='royalblue')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Rating')
plt.ylabel('Predicted Rating')
plt.title('Actual vs Predicted Ratings')
plt.grid(True)
plt.show()

In [ ]:
residuals = y_test - preds
plt.figure(figsize=(10, 6))
sns.histplot(residuals, kde=True, bins=50, color='purple')
plt.title('Distribution of Residuals (Errors)')
plt.xlabel('Residual (Actual - Predicted)')
plt.ylabel('Frequency')
plt.axvline(x=0, color='r', linestyle='--')
plt.show()

### 2. Feature Importance

In [ ]:
importances = model.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(8, 5))
sns.barplot(x=importances[indices], y=np.array(features)[indices], palette='viridis')
plt.title('Random Forest Feature Importances')
plt.xlabel('Relative Importance')
plt.show()

### 3. Classification Metrics (Simulated)
Since this is a Regressor, we frame it as classification by asking: *Did the player improve over their last game?*

In [ ]:
y_test_class = (y_test > X_test['last_game']).astype(int)
preds_class = (preds > X_test['last_game']).astype(int)

print('--- IMPROVEMENT PREDICTION METRICS ---')
print(f'Accuracy:  {accuracy_score(y_test_class, preds_class):.4f}')
print(f'Precision: {precision_score(y_test_class, preds_class):.4f}')
print(f'Recall:    {recall_score(y_test_class, preds_class):.4f}')
print(f'F2 Score:  {fbeta_score(y_test_class, preds_class, beta=2):.4f}')

In [ ]:
cm = confusion_matrix(y_test_class, preds_class)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Regressed/Same', 'Improved'], 
            yticklabels=['Regressed/Same', 'Improved'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix: Player Improvement')
plt.show()